In [19]:
import re
import json
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import StaleElementReferenceException


In [20]:
def login(driver):
    driver.get("https://accounts.google.com/signin")

    # === Enter Email ===
    time.sleep(10)
    email_input = driver.find_element(By.XPATH, "//input[@type='email']")
    email_input.send_keys("i230532@isb.nu.edu.pk")
    email_input.send_keys(Keys.RETURN)

    # === Enter Password ===
    time.sleep(10)
    password_input = driver.find_element(By.XPATH, "//input[@type='password']")
    password_input.send_keys("I$b2341742")
    password_input.send_keys(Keys.RETURN)
    

In [21]:
# def fill(driver):
#     form_data = {
#         "Name": "Taha Ahmed",
#         "Email": "taha.ahmed@example.com",
#         "Contact No.": "1234567890",
#         "Comments": "This is a test comment"
#     }
#     # === Go to Google Form ===
#     time.sleep(10)
#     driver.get("https://docs.google.com/forms/d/e/1FAIpQLSdiDl_sgvqMFtOv45fUCVegya6IbqattwhpgFAe9im3PmD_Xw/viewform")
#     WebDriverWait(driver, 10).until(
#             EC.presence_of_element_located((By.CLASS_NAME, "geS5n"))
#         )
        
#         # Find all question containers with class 'geS5n'
#     question_containers = driver.find_elements(By.CLASS_NAME, "geS5n")
    
#     for container in question_containers:
#         # Extract the label (first text in the question title)
#         try:
#             label = container.text.strip()
#             label = label.replace("*", "").strip()
#             first_line = label.split('\n')[0]  # Get only the first line
#             print(container)
#             print(first_line)
#             input_field = None
#             try:
#                 input_field = container.find_element(By.TAG_NAME, "input")
#             except:
#                 try:
#                     input_field = container.find_element(By.TAG_NAME, "textarea")
#                 except:
#                     continue  # Skip if no input or textarea is found

#             # Match the label with form_data keys and fill the field
#             for key in form_data:
#                 if key.lower() in label.lower():  # Case-insensitive matching
#                     input_field.send_keys(form_data[key])
#                     print(f"Filled '{key}' with '{form_data[key]}'")
#                     break  # Exit loop once the field is filled
#         except Exception as e:
#             print(f"Error processing field with label '{label}': {e}")
#             continue

#     time.sleep(1)


#     # Close the browser
#     # driver.quit()
    

In [22]:
def fill(driver):
    """
    Fills out a Google Form using a generalized approach with regex to skip placeholders.

    This function finds all <input> and <textarea> fields, traverses up the DOM to find
    their labels, and uses a regular expression to ignore common instructional text
    like "Your answer", making the label detection more accurate and robust.
    """
    form_data = {
        "Name": "Taha Ahmed",
        "Email": "taha.ahmed@example.com",
        "Contact No.": "1234567890",
        "Comments": "This is a test comment"
    }
    filled_fields = set() # To track which labels have already been filled
    time.sleep(10)
    # === Go to Google Form ===
    driver.get("https://docs.google.com/forms/d/e/1FAIpQLSdiDl_sgvqMFtOv45fUCVegya6IbqattwhpgFAe9im3PmD_Xw/viewform")
    
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.TAG_NAME, "input"))
    )
    
    # Regex to identify and skip common placeholder/instructional text.
    # It's case-insensitive and matches phrases like "Your answer", "Enter your text", etc.
    placeholder_regex = re.compile(
        r'^(your|enter|my|the) (answer|response|text|name|phone number|comment)', 
        re.IGNORECASE
    )

    # Find all input and textarea elements
    input_fields = driver.find_elements(By.TAG_NAME, "input")
    textarea_fields = driver.find_elements(By.TAG_NAME, "textarea")
    all_fields = input_fields + textarea_fields

    for field in all_fields:
        # Skip buttons and other non-text input types
        try:
            if field.get_attribute('type') not in ['text', 'email', 'tel', 'url', None]: # 'None' for textareas
                 continue
        except StaleElementReferenceException:
            continue

        parent_element = field
        try:
            # Traverse up a maximum of 5 parent levels to find the question label
            for _ in range(10):
                parent_element = parent_element.find_element(By.XPATH, "..")
                raw_text = parent_element.text.strip()
                print(f"Raw text from parent: '{raw_text}'", "Element " + str(parent_element))
                
                if raw_text:
                    # The question is typically the first line of text
                    potential_label = raw_text.split('\n')[0].replace("*", "").strip()

                    # Use regex to check if the found text is a placeholder
                    if placeholder_regex.match(potential_label):
                        print(f"Skipping placeholder text: '{potential_label}'")
                        continue # Skip this text and check the next parent up

                    # Check if this label matches a key in our data and hasn't been filled
                    if potential_label in form_data and potential_label not in filled_fields:
                        field.send_keys(form_data[potential_label])
                        print(f"Filled '{potential_label}' with '{form_data[potential_label]}'")
                        filled_fields.add(potential_label) # Mark as filled
                        break # Exit inner loop and move to the next field
        except StaleElementReferenceException:
            print("Encountered a stale element, skipping.")
            continue
        except Exception as e:
            # Silently continue if an error occurs while processing a field
            # print(f"Could not process a field. Error: {e}")
            continue
                
    time.sleep(1)

    # Close the browser
    # driver.quit()

In [23]:
if __name__ == '__main__':
    option = webdriver.ChromeOptions()
    option.add_argument("--incognito")
    # option.add_argument("--headless")


    browser = webdriver.Chrome(options = option)
    login(browser)
    fill(browser)

Raw text from parent: 'Your answer' Element <selenium.webdriver.remote.webelement.WebElement (session="38da06a9a1b9dc17bd3fe339b835b305", element="f.5AABAA3E7C793A70D00C30943C7C4D2B.d.71325DD2DD61C411F459E7CB4C8B3F88.e.127")>
Skipping placeholder text: 'Your answer'
Raw text from parent: 'Your answer' Element <selenium.webdriver.remote.webelement.WebElement (session="38da06a9a1b9dc17bd3fe339b835b305", element="f.5AABAA3E7C793A70D00C30943C7C4D2B.d.71325DD2DD61C411F459E7CB4C8B3F88.e.126")>
Skipping placeholder text: 'Your answer'
Raw text from parent: 'Your answer' Element <selenium.webdriver.remote.webelement.WebElement (session="38da06a9a1b9dc17bd3fe339b835b305", element="f.5AABAA3E7C793A70D00C30943C7C4D2B.d.71325DD2DD61C411F459E7CB4C8B3F88.e.125")>
Skipping placeholder text: 'Your answer'
Raw text from parent: 'Your answer' Element <selenium.webdriver.remote.webelement.WebElement (session="38da06a9a1b9dc17bd3fe339b835b305", element="f.5AABAA3E7C793A70D00C30943C7C4D2B.d.71325DD2DD61C41